In [ ]:
# Load skincare products with Logistic Regression predictions and K-Means clusters.
import pandas as pd
products = pd.read_csv("skincare_with_predictions.csv")
clusters = pd.read_csv("skincare_clustered.csv")
print("Total Products:", len(products))
print("Total Clustered Products:", len(clusters))
print("Number of Clusters:", clusters["Cluster"].nunique())
print("\nSame Product Order:", products[["brand", "name", "ingredients_clean"]].equals(clusters[["brand", "name", "ingredients_clean"]] ))


In [ ]:
# Combine skin-type predictions and K-Means cluster numbers into one dataset.
if products[["brand", "name", "ingredients_clean"]].equals( clusters[["brand", "name", "ingredients_clean"]]):
    products["Cluster"] = clusters["Cluster"].values

    print("Datasets Combined Successfully!")
    print("Total Products:", len(products))
    print("Total Clusters:", products["Cluster"].nunique())

    display(products[["brand", "name", "category", "price_pkr", "Cluster"]].head())
else:
    print("Product order does not match. Datasets were not combined.")


In [ ]:
# Combine original and predicted skin-type labels for product recommendations.
skin_cols = ["tag_dry", "tag_oily", "tag_combination", "tag_normal", "tag_sensitive", "tag_acne_prone"]
for skin in skin_cols:
    new_col = "match_" + skin

    products[new_col] = products[skin].fillna(0).astype(int)

    products.loc[products["has_skin_label"] == 0, new_col] = products.loc[products["has_skin_label"] == 0, "pred_" + skin ].astype(int)

print("Skin-Type Labels Prepared Successfully!")

match_cols = ["match_" + skin for skin in skin_cols]

print("\nProducts Matching Each Skin Type:")
print(products[match_cols].sum())


In [ ]:
# Filter skincare products according to skin type, category, and budget.
print("Available Categories:")
print(products["category"].dropna().unique())
skin_type = input("\nEnter skin type: ").strip().lower()
category = input("Enter category (or All): ").strip()
budget = float(input("Enter maximum budget in PKR: "))
skin_column = "match_tag_" + skin_type
if skin_column in products.columns:
    filtered_products = products[
        (products[skin_column] == 1) &
        (products["price_pkr"] <= budget)].copy()

    if category.lower() != "all":
        filtered_products = filtered_products[
            filtered_products["category"].str.lower() == category.lower()
        ]

    print("\nMatching Products:", len(filtered_products))

    display(filtered_products[
        ["brand", "name", "category", "price_pkr", "Cluster"]
    ].head(10))

else:
    print("Invalid skin type!")


In [ ]:
# Recommend 5 products with ingredients similar to a selected product.
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity
X_tfidf = load_npz("ingredients_tfidf.npz")
product_index = int(input("Enter a product index from the table: "))
if product_index in filtered_products.index:
    similarity = cosine_similarity( X_tfidf[product_index], X_tfidf[filtered_products.index])[0]
    recommendations = filtered_products.copy()
    recommendations["Similarity"] = similarity
    recommendations = recommendations.drop(product_index)
    recommendations = recommendations.sort_values("Similarity", ascending=False )
    print("\nTop 5 Similar Products:")
    display(recommendations[
        ["brand", "name", "category", "price_pkr", "Similarity", "Cluster"]].head(5))
else:
    print("Invalid product index!")


In [ ]:
# Display common ingredients between the selected product and its recommendations.
selected_ingredients = set(products.loc[product_index, "ingredients_clean"].split("|"))
print("Selected Product:", products.loc[product_index, "name"])
for i, row in recommendations.head(5).iterrows():
    other_ingredients = set(row["ingredients_clean"].split("|"))
    common = selected_ingredients.intersection(other_ingredients)
    print("\nRecommended Product:", row["name"])
    print("Common Ingredients:", len(common))
    print(", ".join(sorted(common)[:10]))


In [ ]:
# Check the selected product's important ingredients and compare them with recommendations.
# The dataset has 6 active-ingredient flags; 4 more are created here from the ingredient lists.
ing = products["ingredients_clean"].str.lower()
products["has_retinoid"] = ing.str.contains(r"retinol|retinal|retinyl|adapalene|bakuchiol").astype(int)
products["has_vitamin_c"] = ing.str.contains(r"ascorbic|ascorbyl").astype(int)
products["has_aha"] = ing.str.contains(r"glycolic|lactic acid|mandelic").astype(int)
products["has_uv_filter"] = ing.str.contains(
    r"zinc oxide|titanium dioxide|avobenzone|octinoxate|homosalate|octocrylene|octisalate|oxybenzone").astype(int)
recommendations[["has_retinoid", "has_vitamin_c", "has_aha", "has_uv_filter"]] = \
    products.loc[recommendations.index, ["has_retinoid", "has_vitamin_c", "has_aha", "has_uv_filter"]]

active_cols = [
    "has_niacinamide",
    "has_zinc",
    "has_salicylic_acid",
    "has_ceramide",
    "has_hyaluronic",
    "has_squalane",
    "has_retinoid",
    "has_vitamin_c",
    "has_aha",
    "has_uv_filter"
]

print("Selected Product:", products.loc[product_index, "name"])

print("\nSelected Product's Ingredient Features:")
print(products.loc[product_index, active_cols])

print("\nRecommended Products' Ingredient Features:")
display(recommendations[["name"] + active_cols].head(5))

In [ ]:
# Rank products with one combined score: cosine similarity carries most of the weight, and each shared
# important ingredient adds a small bonus. (Sorting by shared ingredients first would let one common flag,
# such as hyaluronic acid in ~45% of products, push weak matches above much more similar products.)
selected_actives = []
for col in active_cols:
    if products.loc[product_index, col] == 1:
        selected_actives.append(col)

print("Selected Product's Important Ingredients:")
print(selected_actives)

recommendations["Shared_Actives"] = recommendations[selected_actives].sum(axis=1) if selected_actives else 0
recommendations["Score"] = recommendations["Similarity"] + 0.05 * recommendations["Shared_Actives"]

recommendations = recommendations.sort_values("Score", ascending=False)

print("\nTop 5 Improved Recommendations:")

display(recommendations[
    ["brand", "name", "price_pkr", "Shared_Actives", "Similarity", "Score", "Cluster"]
].head(5))

In [ ]:

# Show alternatives that share at least one important ingredient with the selected product.

active_matches = recommendations[
    recommendations["Shared_Actives"] > 0
].copy()

print("Products Sharing Important Ingredients:", len(active_matches))

if len(active_matches) > 0:
    print("\nTop 5 Active-Ingredient Alternatives:")

    display(active_matches[
        ["brand", "name", "price_pkr", "Shared_Actives", "Similarity", "Score", "Cluster"]
    ].head(5))

else:
    print("No matching active-ingredient alternatives found within the selected filters.")


In [ ]:
# Check niacinamide and zinc ingredients in the selected product and its alternatives.
check_products = products.loc[
    [product_index] + active_matches.head(5).index.tolist()
]

for i, row in check_products.iterrows():

    print("\nProduct:", row["name"])

    print("Niacinamide Flag:", row["has_niacinamide"])
    print("Zinc Flag:", row["has_zinc"])

    print("Matching Ingredients:")

    for ingredient in row["ingredients_clean"].split("|"):
        if "niacinamide" in ingredient or "zinc" in ingredient:
            print(ingredient)


In [ ]:
# Final recommendation function.
# Score = cosine similarity + 0.05 per shared important ingredient + 0.03 if in the same K-Means cluster.
# Products with similarity below min_similarity are not shown, because they share almost nothing except
# basic ingredients like water and glycerin. Sun-protection results must be real sunscreens (not
# self-tanners); many sunscreens do not list their UV filters in ingredients_clean, so names are checked.

def recommend_products(product_index, skin_type, category, budget, top_n=5, min_similarity=0.08):

    skin_column = "match_tag_" + skin_type.lower()

    if skin_column not in products.columns or product_index not in products.index:
        print("Invalid skin type or product index!")
        return pd.DataFrame()

    matches = products[
        (products[skin_column] == 1) &
        (products["price_pkr"] <= budget)
    ].copy()

    if category.lower() != "all":
        matches = matches[matches["category"].str.lower() == category.lower()]

    if category.lower() == "sun protection":
        is_sunscreen = matches["name"].str.contains("SPF|sunscreen|sunblock|broad spectrum|UV",
                                                    case=False, na=False) | (matches["has_uv_filter"] == 1)
        is_tanner = matches["name"].str.contains("tan|bronz", case=False, na=False)
        matches = matches[is_sunscreen & ~is_tanner]

    matches = matches.drop(index=product_index, errors="ignore")

    if len(matches) == 0:
        print("No products found within the selected filters.")
        return pd.DataFrame()

    matches["Similarity"] = cosine_similarity(X_tfidf[product_index], X_tfidf[matches.index])[0]

    selected_actives = [col for col in active_cols if products.loc[product_index, col] == 1]
    matches["Shared_Actives"] = matches[selected_actives].sum(axis=1) if selected_actives else 0
    matches["Same_Cluster"] = (matches["Cluster"] == products.loc[product_index, "Cluster"]).astype(int)
    matches["Score"] = matches["Similarity"] + 0.05 * matches["Shared_Actives"] + 0.03 * matches["Same_Cluster"]

    matches["Match_Type"] = "General ingredient similarity"
    matches.loc[matches["Shared_Actives"] > 0, "Match_Type"] = "Shared key ingredients"

    matches = matches[matches["Similarity"] >= min_similarity]

    if len(matches) == 0:
        print(f"No sufficiently similar products (similarity >= {min_similarity}) within the selected filters.")
        return pd.DataFrame()

    matches = matches.sort_values("Score", ascending=False)

    return matches[
        ["brand", "name", "category", "price_pkr",
         "Shared_Actives", "Similarity", "Score", "Match_Type", "Cluster"]
    ].head(top_n)


# Test with a sunscreen as the reference product.
print("Selected Product:", products.loc[2606, "name"])
results3 = recommend_products(2606, "combination", "Sun protection", 10000)
display(results3)

In [ ]:

# Test the recommendation function with a different product and skin type.

print("Selected Product:", products.loc[3, "name"])

results2 = recommend_products(3, "dry", "Moisturizer", 6000)

print("\nRecommended Products:")
display(results2)


In [ ]:

# Test recommendations for different skin types and product categories.

print("Sensitive Skin - Cleanser")
display(recommend_products(6, "sensitive", "Cleanser", 6000))

print("\nAcne-Prone Skin - Treatment")
display(recommend_products(6, "acne_prone", "Treatment", 6000))

print("\nCombination Skin - Sun Protection")
display(recommend_products(2606, "combination", "Sun protection", 10000))


In [ ]:
# Check how many sun-protection products match the skin type and budget.

sun_matches = products[
    (products["match_tag_combination"] == 1) &
    (products["category"].str.lower() == "sun protection") &
    (products["price_pkr"] <= 6000)
]

print("Sun-protection products matching skin type and budget:", len(sun_matches))
print("Of these, self-tanners (excluded from sunscreen recommendations):",
      int(sun_matches["name"].str.contains("tan|bronz", case=False).sum()))

display(sun_matches[
    ["brand", "name", "category", "price_pkr", "has_uv_filter", "Cluster"]
].head(10))

In [ ]:

# Test sunscreen recommendations using a sunscreen as the reference product.
print("Selected Product:", products.loc[2606, "name"])
sunscreen_results = recommend_products(
    2606, "combination", "Sun protection", 6000
)
# With a strict budget there may be no sufficiently similar sunscreens; an empty result
# is better than recommending products that share almost no ingredients.

print("\nRecommended Sunscreen Alternatives:")
display(sunscreen_results)


In [ ]:

# Save the final SkinMatch dataset containing skin-type matches and K-Means clusters.

products.to_csv("skinmatch_recommendations.csv", index=False)

print("Recommendation Dataset Saved Successfully!")
print("Total Products:", len(products))
print("Total Clusters:", products["Cluster"].nunique())
print("Total Skin Types:", len(skin_cols))


### Conclusion

In this phase, the SkinMatch recommendation system was developed by combining Logistic Regression skin-type
predictions, K-Means clusters and ingredient similarity.

The dataset contains **2,724 skincare products**, six skin-type matching labels (original labels where available,
otherwise Logistic Regression predictions) and ten ingredient-based clusters.

Products are filtered by:

- **Skin type:** dry, oily, combination, normal, sensitive or acne-prone.
- **Product category:** serum, moisturizer, cleanser, treatment, sun protection, eye care or mask.
- **Budget:** maximum price in PKR.

The remaining products are ranked with one score:

- **Cosine similarity** between TF-IDF ingredient vectors (main part of the score),
- **+0.05** for each shared important ingredient (niacinamide, zinc, salicylic acid, ceramides, hyaluronic acid,
  squalane, retinoids, vitamin C, AHAs, UV filters),
- **+0.03** if the product is in the same K-Means cluster.

Products with a similarity below **0.08** are not recommended, because they share little beyond common base
ingredients. Sun-protection recommendations only include real sunscreens (self-tanners are excluded). When no product is
similar enough, the system returns no results instead of weak matches.

The threshold of 0.08 was chosen by inspecting results: relevant alternatives (for example sulfur and
niacinamide acne treatments) scored about 0.08–0.10, while unrelated products sharing only base ingredients
scored below 0.08.

**Limitations:** ingredient similarity ignores concentrations and ingredient order and does not guarantee equal
benefits or suitability. Some category labels in the source data are imprecise (for example toners listed as
cleansers), and PKR prices for imported brands are converted from US prices rather than verified retail prices.

The recommendation dataset was saved as **`skinmatch_recommendations.csv`** for the Streamlit application.